### Count Velocity Obs

In [6]:
import xarray as xr

filepath_2012 = '/data/SO3/edavenport/tpose6/tao_profiles/TAO_WO_2012_ADCP_v2.nc'
filepath_2013 = '/data/SO3/edavenport/tpose6/tao_profiles/TAO_WO_2013_ADCP.nc'

ds_adcp = xr.concat([xr.open_dataset(filepath_2012), xr.open_dataset(filepath_2013)], dim='iPROF')

filepath_2012 = '/data/SO3/edavenport/tpose6/tao_profiles/TAO_WO_2012_CUR_v2.nc'
filepath_2013 = '/data/SO3/edavenport/tpose6/tao_profiles/TAO_WO_2013_CUR.nc'

ds_cur = xr.concat([xr.open_dataset(filepath_2012), xr.open_dataset(filepath_2013)], dim='iPROF')

/tmp/ipykernel_1653893/2532352644.py:6: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_adcp = xr.concat([xr.open_dataset(filepath_2012), xr.open_dataset(filepath_2013)], dim='iPROF')
/tmp/ipykernel_1653893/2532352644.py:11: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_cur = xr.concat([xr.open_dataset(file

In [8]:
total_adcp_U = ds_adcp['prof_U'].count()
total_adcp_V = ds_adcp['prof_V'].count()
total_cur_U = ds_cur['prof_U'].count()
total_cur_V = ds_cur['prof_V'].count()

total = total_adcp_U + total_adcp_V + total_cur_U + total_cur_V

percent_cur = (total_cur_U + total_cur_V) / total * 100
percent_adcp = (total_adcp_U + total_adcp_V) / total * 100
print(f"percent ADCP profiles: {total_adcp_U + total_adcp_V} ({percent_adcp:.2f}%)")
print(f"percent CUR profiles: {total_cur_U + total_cur_V} ({percent_cur:.2f}%)")

percent ADCP profiles: <xarray.DataArray ()> Size: 8B
array(8724640) (83.78%)
percent CUR profiles: <xarray.DataArray ()> Size: 8B
array(1688640) (16.22%)


### Count SSH Obs

In [1]:
## Read in ITER 0 Diagnostics 
import matplotlib.pyplot as plt
import cmocean.cm as cmo
import xarray as xr
from xmitgcm import open_mdsdataset
import numpy as np
import xarray as xr

## it doesnt matter what this is, we just want the grid in ds format
grid_dir = '/data/SO6/TPOSE_diags/tpose6/grid_6/'
data_dir = '/data/SO3/edavenport/tpose6/sep2012/velocity_assim/run_iter22/'

prefix = ['diag_state']
ds = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=72,prefix=prefix,ref_date='2012-09-01',delta_t=1200)
ds['XC'] = ds.XC.astype(float)
ds['YC'] = ds.YC.astype(float)
ds['Z'] = ds.Z.astype(float)
ds['XG'] = ds.XG.astype(float)
ds['YG'] = ds.YG.astype(float)

In [16]:
import numpy as np
import pandas as pd

# File paths for both years
rads_ja_2012 = '/data/SO6/TPOSE/constraints/rads/rads_ja_tpose6_QC_2012'
rads_c2_2012 = '/data/SO6/TPOSE/constraints/rads/rads_c2_tpose6_QC_2012'
rads_ja_2013 = '/data/SO6/TPOSE/constraints/rads/rads_ja_tpose6_QC_2013'
rads_c2_2013 = '/data/SO6/TPOSE/constraints/rads/rads_c2_tpose6_QC_2013'

def load_rads(filename, nx, ny, nt):
    with open(filename, 'rb') as f:
        data = np.fromfile(f, dtype='>f4')
    data = data.reshape((nx, ny, nt), order='F').transpose(1, 0, 2).astype(float)
    data[data == -9999.0] = np.nan
    return data / 100  # cm to m

nx, ny = len(ds.XC), len(ds.YC)

# Load and concatenate both years (2012: 366 days leap, 2013: 365 days)
rads_ja = np.concatenate([load_rads(rads_ja_2012, nx, ny, 366),
                           load_rads(rads_ja_2013, nx, ny, 365)], axis=2)
rads_c2 = np.concatenate([load_rads(rads_c2_2012, nx, ny, 366),
                           load_rads(rads_c2_2013, nx, ny, 365)], axis=2)

# Daily date coordinates spanning 2012 + 2013
dates = pd.date_range('2012-01-01', periods=366 + 365, freq='D')

rads_ja_da = xr.DataArray(rads_ja, dims=['YC', 'XC', 'time'],
                           coords={'YC': ds.YC, 'XC': ds.XC, 'time': dates}, name='rads_ja')
rads_c2_da = xr.DataArray(rads_c2, dims=['YC', 'XC', 'time'],
                           coords={'YC': ds.YC, 'XC': ds.XC, 'time': dates}, name='rads_c2')

In [18]:
### Count SSH obs (non-NaN) for each 4-month assimilation window
periods = {
    'Sep-Dec 2012':      ('2012-09-01', '2012-12-31'),
    'Nov 2012-Feb 2013': ('2012-11-01', '2013-02-28'),
    'Jan-Apr 2013':      ('2013-01-01', '2013-04-30'),
    'Mar-Jun 2013':      ('2013-03-01', '2013-06-30'),
}

print(f"{'Period':<22}  {'Jason':>12}  {'CryoSat':>12}  {'Jason %':>9}  {'CryoSat %':>10}")
print("-" * 72)
for label, (start, end) in periods.items():
    ja_count = rads_ja_da.sel(time=slice(start, end)).count().item()
    c2_count = rads_c2_da.sel(time=slice(start, end)).count().item()
    total = ja_count + c2_count
    print(f"{label:<22}  {ja_count:>12,}  {c2_count:>12,}  {ja_count/total*100:>8.1f}%  {c2_count/total*100:>9.1f}%")

Period                         Jason       CryoSat    Jason %   CryoSat %
------------------------------------------------------------------------
Sep-Dec 2012                 992,377       456,152      68.5%       31.5%
Nov 2012-Feb 2013            740,827       447,784      62.3%       37.7%
Jan-Apr 2013                 455,115       443,459      50.6%       49.4%
Mar-Jun 2013                 465,152       460,734      50.2%       49.8%


### SST Obs

In [ ]:
# OISST_filename = '/data/SO6/TPOSE/constraints/sst/mw_fusion_tpose6_2012'

# # Read the binary file (big-endian float32)
# with open(OISST_filename, 'rb') as f:
#     OISST = np.fromfile(f, dtype='>f4')

# # Define dimensions
# nx, ny, nt = len(ds.XC), len(ds.YC), 366

# # Reshape (Fortran order to match MATLAB)
# OISST = OISST.reshape((nx, ny, nt), order='F')
# OISST = OISST.transpose(1,0,2)

# # Convert to xarray DataArray
# OISST_da = xr.DataArray(
#     OISST[:,:,-len(ds.time):],  # Align time dimension
#     dims=['YC', 'XC', 'time'],
#     coords={'YC': ds.YC, 'XC': ds.XC, 'time': ds.time},
#     name='OISST'
# )

# # SST weights
# uncertainty_file = '/data/SO6/TPOSE/model_setup/linked_files/TP6/SST_error_tpose6_17Sto17N.bin'

# # Read the binary file (big-endian float32)
# with open(uncertainty_file, 'rb') as f:
#     weights = np.fromfile(f, dtype='>f4')

# # Define dimensions
# nx, ny = len(ds.XC), len(ds.YC)

# # Reshape (Fortran order to match MATLAB)
# weights = weights.reshape((nx, ny), order='F')
# weights = weights.transpose(1,0)

# # Convert to xarray DataArray
# weights_da = xr.DataArray(
#     weights,  # Align time dimension
#     dims=['YC', 'XC'],
#     coords={'YC': ds.YC, 'XC': ds.XC},
#     name='weights'
# )